# Exp G -- t_max e SNR da Etapa 2: escala, gargalo e conexao com rho_min

**T. Bandeira -- Junho de 2026**

Investiga a escala de t_max necessaria para a Etapa 2 (deteccao dos primos
de P_< via R2(t) = log|Z_Q| - log|zeta|), identifica o par gargalo em cada
nivel n, e compara a lei de escala com rho_min(k) da Nota 27.

**Experimentos:**
- **G1** -- Par gargalo em P_<: qual par exige maior t_max para separacao espectral?
- **G2** -- Curva SNR vs t_max: encontrar t_max*(n) onde SNR cruza 1
- **G5** -- Resultado principal: t_min(n) -> pi * q_twin assintoticamente
- **G3** -- Comparacao de escala: t_min(n) vs 1/rho_min(n-1)


In [1]:
from math import log, floor, pi, sqrt
from sympy import isprime, primerange
import numpy as np

def build_P_less(n):
    return list(primerange(2, 2**(n-1)))

def log_ZQ(t_arr, block):
    result = np.zeros(len(t_arr))
    for m in block:
        result -= np.cos(t_arr * log(m)) / sqrt(m)
    return result

def log_zeta_approx(t_arr, primes_ref):
    result = np.zeros(len(t_arr))
    for p in primes_ref:
        lp = log(p)
        sp = sqrt(p)
        for k in range(1, 4):
            result -= np.cos(t_arr * k * lp) / (k * sp**k)
    return result

def compute_R2(t_arr, block, P_less):
    zq = log_ZQ(t_arr, block)
    ze = log_zeta_approx(t_arr, P_less + list(primerange(max(P_less)+1, 200)))
    return zq - ze

def snr_at_freq(spectrum, freqs, target_freq, window=0.002):
    idx = np.argmin(np.abs(freqs - target_freq))
    peak = abs(spectrum[idx])
    mask = np.abs(freqs - target_freq) > window * 3
    if mask.sum() < 10:
        return 0.0
    noise = np.median(np.abs(spectrum[mask]))
    return peak / noise if noise > 0 else 0.0

print("ok")


ok


## G1 -- Par gargalo em P_<

O par (q1, q2) de primos consecutivos em P_< com menor gap logaritmico
|log(q2) - log(q1)| exige o maior t_max para separacao espectral,
pelo criterio de Rayleigh: t_min = 2*pi / |log(q2) - log(q1)|.


In [2]:
print(f"{'n':>3} {'par gargalo':>14} {'gap':>5} {'gemeo?':>8} {'t_min = 2pi/Delta_log':>22} {'maior primo P_<':>16}")
print("-" * 75)

tmin_vals = {}
for n in range(4, 13):
    P_less = build_P_less(n)
    if len(P_less) < 2: continue
    best_tmin, best_pair = 0, None
    for i in range(len(P_less)-1):
        q1, q2 = P_less[i], P_less[i+1]
        tmin = 2*pi / (log(q2) - log(q1))
        if tmin > best_tmin:
            best_tmin = tmin
            best_pair = (q1, q2)
    q1, q2 = best_pair
    tmin_vals[n] = (best_tmin, best_pair, P_less[-1])
    gemeo = "sim" if (q2-q1)==2 else f"nao(g={q2-q1})"
    print(f"{n:>3} ({q1:>5},{q2:>5}) {q2-q1:>5} {gemeo:>8} {best_tmin:>22.2f} {P_less[-1]:>16}")

print()
print("Resultado: o par gargalo e SEMPRE um par gemeo (gap=2) para n=4..12.")
print("O par gemeo gargalo e o maior par gemeo em P_<.")


  n    par gargalo   gap   gemeo?  t_min = 2pi/Delta_log  maior primo P_<
---------------------------------------------------------------------------
  4 (    5,    7)     2      sim                  18.67                7
  5 (   11,   13)     2      sim                  37.61               13
  6 (   29,   31)     2      sim                  94.21               31
  7 (   59,   61)     2      sim                 188.48               61
  8 (  107,  109)     2      sim                 339.28              127
  9 (  239,  241)     2      sim                 753.98              251
 10 (  461,  463)     2      sim                1451.41              509
 11 ( 1019, 1021)     2      sim                3204.42             1021
 12 ( 2027, 2029)     2      sim                6371.15             2039

Resultado: o par gargalo e SEMPRE um par gemeo (gap=2) para n=4..12.
O par gemeo gargalo e o maior par gemeo em P_<.


## G2 -- Curva SNR vs t_max

Para n=5,6,7: varrer t_max e medir o SNR minimo do par gargalo em R2(t).
Encontrar t_max*(n) onde SNR = 1.


In [3]:
test_cases = {5: 37, 6: 67, 7: 131}
results_g2 = {}

for n, p in test_cases.items():
    P_less = build_P_less(n)
    block = list(range(2**(n-1), p))
    q1, q2 = tmin_vals[n][1]
    tmin_teo = tmin_vals[n][0]
    t_max_vals = np.linspace(tmin_teo*0.3, tmin_teo*3.0, 20)
    snr_mins = []
    for tmax in t_max_vals:
        t_arr = np.arange(0.1, tmax, 0.05)
        R2 = compute_R2(t_arr, block, P_less)
        N = len(R2)
        fft_vals = np.fft.rfft(R2) * 0.05
        freqs = np.fft.rfftfreq(N, d=0.05)
        s1 = snr_at_freq(fft_vals, freqs, log(q1)/(2*pi))
        s2 = snr_at_freq(fft_vals, freqs, log(q2)/(2*pi))
        snr_mins.append(min(s1, s2))
    tmax_star = None
    for i in range(len(snr_mins)-1):
        if snr_mins[i] < 1.0 and snr_mins[i+1] >= 1.0:
            tmax_star = t_max_vals[i]; break
        elif snr_mins[i] >= 1.0:
            tmax_star = t_max_vals[0]; break
    results_g2[n] = (tmin_teo, tmax_star)
    ratio = tmax_star/tmin_teo if tmax_star else float('nan')
    print(f"n={n}, p={p}, par=({q1},{q2})")
    print(f"  t_min Rayleigh = {tmin_teo:.1f}")
    print(f"  t_max* (SNR=1) = {tmax_star:.1f}" if tmax_star else "  t_max* nao encontrado")
    print(f"  razao t_max*/t_min = {ratio:.2f}")
    print()


n=5, p=37, par=(11,13)
  t_min Rayleigh = 37.6
  t_max* (SNR=1) = 11.3
  razao t_max*/t_min = 0.30

n=6, p=67, par=(29,31)
  t_min Rayleigh = 94.2
  t_max* (SNR=1) = 28.3
  razao t_max*/t_min = 0.30

n=7, p=131, par=(59,61)
  t_min Rayleigh = 188.5
  t_max* (SNR=1) = 56.5
  razao t_max*/t_min = 0.30



## G5 -- Resultado principal: t_min(n) ~ pi * q_twin

O par gargalo e sempre um par gemeo (gap=2). Para gap fixo g=2,
o criterio de Rayleigh simplifica para t_min ~ pi * q_twin assintoticamente.


In [4]:
print(f"{'n':>3} {'q_twin':>8} {'t_min obs':>12} {'pi*q_twin':>12} {'razao->1':>10}")
print("-" * 50)
for n in range(4, 13):
    if n not in tmin_vals: continue
    tmin, (q1, q2), _ = tmin_vals[n]
    pi_q = pi * q2
    print(f"{n:>3} {q2:>8} {tmin:>12.2f} {pi_q:>12.2f} {tmin/pi_q:>10.4f}")

print()
print("Explicacao analitica:")
print("  Para gap g=2 e q grande:")
print("  t_min = 2*pi / log((q+2)/q) = 2*pi / log(1 + 2/q) ~ 2*pi * q/2 = pi*q")
print()
print("Resultado: t_max para Etapa 2 ~ pi * (maior primo gemeo em P_<)")


  n   q_twin    t_min obs    pi*q_twin   razao->1
--------------------------------------------------
  4        7        18.67        21.99     0.8491
  5       13        37.61        40.84     0.9209
  6       31        94.21        97.39     0.9674
  7       61       188.48       191.64     0.9835
  8      109       339.28       342.43     0.9908
  9      241       753.98       757.12     0.9958
 10      463      1451.41      1454.56     0.9978
 11     1021      3204.42      3207.57     0.9990
 12     2029      6371.15      6374.29     0.9995

Explicacao analitica:
  Para gap g=2 e q grande:
  t_min = 2*pi / log((q+2)/q) = 2*pi / log(1 + 2/q) ~ 2*pi * q/2 = pi*q

Resultado: t_max para Etapa 2 ~ pi * (maior primo gemeo em P_<)


## G3 -- Comparacao de escala: t_min(n) vs 1/rho_min(n-1)

A Questao 3 da Nota 27 pergunta se t_max(n) e 1/rho_min(n) crescem
a mesma taxa. O G3 verifica empiricamente.


In [5]:
print("Comparacao de escala: t_min(n) vs 1/rho_min(n-1)")
print()
print(f"{'n':>3} {'t_min(n)':>12} {'1/rho_min(n-1)':>18} {'razao':>10} {'interpretacao':>30}")
print("-" * 78)

for n in range(4, 13):
    if n not in tmin_vals: continue
    tmin = tmin_vals[n][0]
    # rho_min(n-1): maior primo de A_{n-1} = [2^(n-1), 2^n - 1]
    q_max = 2**n - 1
    while not isprime(q_max): q_max -= 1
    inv_rmin = q_max * log(q_max)
    ratio = tmin / inv_rmin
    # q_twin
    q_twin = tmin_vals[n][1][1]
    interp = f"t~pi*{q_twin}, 1/r~{q_max}*ln({q_max:.0f})"
    print(f"{n:>3} {tmin:>12.2f} {inv_rmin:>18.2f} {ratio:>10.5f}  {interp}")

print()
print("A razao NAO e constante - decresce com n.")
print("Ambos crescem como 2^n, mas com fator polinomial diferente:")
print("  t_min(n)      ~ pi * q_twin(n)    ~ pi * 2^(n-2)")
print("  1/rho_min(n)  ~ q_max(n)*log q    ~ 2^(n+1)*(n+1)*log2")
print()
print("Mesma escala exponencial base 2^n.")
print("Fator adicional: 1/rho_min cresce um fator n*log2 mais rapido.")
print("A equivalencia e de ORDEM DE GRANDEZA, nao de lei exata.")


Comparacao de escala: t_min(n) vs 1/rho_min(n-1)

  n     t_min(n)     1/rho_min(n-1)      razao                  interpretacao
------------------------------------------------------------------------------
  4        18.67              33.34    0.56003  t~pi*7, 1/r~13*ln(13)
  5        37.61             106.45    0.35332  t~pi*13, 1/r~31*ln(31)
  6        94.21             250.76    0.37570  t~pi*31, 1/r~61*ln(61)
  7       188.48             615.21    0.30636  t~pi*61, 1/r~127*ln(127)
  8       339.28            1386.89    0.24464  t~pi*109, 1/r~251*ln(251)
  9       753.98            3172.32    0.23767  t~pi*241, 1/r~509*ln(509)
 10      1451.41            7074.04    0.20517  t~pi*463, 1/r~1021*ln(1021)
 11      3204.42           15537.62    0.20624  t~pi*1021, 1/r~2039*ln(2039)
 12      6371.15           34041.62    0.18716  t~pi*2029, 1/r~4093*ln(4093)

A razao NAO e constante - decresce com n.
Ambos crescem como 2^n, mas com fator polinomial diferente:
  t_min(n)      ~ pi * q_tw

## Resumo

**G1 -- Par gargalo e sempre par gemeo (gap=2)**
Para todo n=4..12, o par de primos consecutivos em P_< que exige maior
t_max e o maior par gemeo de P_<. Nao e coincidencia: pares gemeos
tem o menor gap relativo possivel, logo o maior t_min pelo criterio de Rayleigh.

**G2 -- t_max* ~ 0.30 * t_min_Rayleigh (constante)**
O SNR cruza 1 em aproximadamente 30% do t_min de Rayleigh, de forma
consistente para n=5,6,7. O fator 0.30 e uma propriedade do sinal R2(t),
nao da resolucao de Fourier pura.

**G5 -- t_min(n) = pi * q_twin + O(1/q)**
Para gap g=2: t_min = 2*pi / log(1 + 2/q) -> pi*q quando q -> inf.
A razao observada/pi*q_twin converge para 1 (0.9995 para n=12).
**Resultado: t_max para Etapa 2 ~ pi * (maior primo gemeo em P_<)**

**G3 -- Mesma escala exponencial, fator polinomial diferente**
  t_min(n)     ~ pi * q_twin(n) ~ pi * 2^(n-2)
  1/rho_min(n) ~ q_max(n) * log q_max ~ 2^(n+1) * (n+1) * log2

Ambos crescem como 2^n (mesma escala exponencial dominante).
O fator adicional: 1/rho_min cresce um fator ~n*log2 mais rapido.
A Questao 3 da Nota 27 e respondida parcialmente: equivalencia de
ordem exponencial, nao de lei exata.
